# 🧠 AI vs Real Image Classifier — v2 (Optimized)

| | |
|---|---|
| **Architecture** | EfficientNetV2-S (faster + more accurate than B3) |
| **Pipeline** | `tf.data` with prefetch, cache, parallel map |
| **Speed** | Mixed precision (FP16) · XLA JIT · ~3–5× faster than v1 |
| **Accuracy** | Target: 80–88 % val accuracy on T4 GPU |
| **Stability** | Auto-resume from checkpoint · crash-safe |

> ⚠️ **Required:** `Runtime → Change runtime type → T4 GPU`  
> ⚠️ **Required:** Upload your dataset zip to Google Drive first


In [ ]:
# ── Step 1: GPU check + dependencies ──────────────────────────────────────
import subprocess, os, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('⚠️  No GPU detected — switch runtime to T4 GPU for best results')

# Install / upgrade key packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'tensorflow', 'scikit-learn', 'matplotlib', 'seaborn',
                'tqdm', 'Pillow'], check=True)

import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

# ── Enable mixed precision (FP16) — biggest single speed-up on T4 ──────────
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print(f'Compute dtype: {mixed_precision.global_policy().compute_dtype}')   # float16
print(f'Variable dtype: {mixed_precision.global_policy().variable_dtype}') # float32

# ── XLA JIT compilation ─────────────────────────────────────────────────────
tf.config.optimizer.set_jit(True)

# ── GPU memory growth ───────────────────────────────────────────────────────
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print('✅ Mixed precision + XLA enabled')


✅ GPU: Tesla T4, 15360 MiB
TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Compute dtype: float16
Variable dtype: float32
✅ Mixed precision + XLA enabled


In [ ]:
# ── Step 2: Global config ──────────────────────────────────────────────────
import os, json, random, io, zipfile, shutil, glob, warnings
import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT   = '/content/dataset'
CHECKPOINT_DIR = '/content/checkpoints'
MODEL_OUT      = '/content/model_v2.keras'
CLASS_JSON     = '/content/class_names_v2.json'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATASET_ROOT,   exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────────
IMG_SIZE    = 224          # EfficientNetV2-S native size
BATCH_SIZE  = 128           # FP16 lets us double batch size vs v1
SEED        = 42
AUTOTUNE    = tf.data.AUTOTUNE

# Phase 1 — head warmup
EPOCHS_P1   = 10
LR_P1       = 1e-3

# Phase 2 — fine-tune top layers
EPOCHS_P2   = 20
LR_P2       = 2e-5

TRAIN_FRAC  = 0.75
VAL_FRAC    = 0.15
THRESHOLD   = 0.60
UNFREEZE_TOP = 50          # unfreeze top-N base layers in Phase 2

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('✅ Config set')
print(f'   Batch size : {BATCH_SIZE}  (FP16 enabled — safe to double)')
print(f'   Image size : {IMG_SIZE}x{IMG_SIZE}')


✅ Config set
   Batch size : 128  (FP16 enabled — safe to double)
   Image size : 224x224


In [ ]:
# ── Step 3: Mount Drive & locate zip ───────────────────────────────────────
from google.colab import drive as gdrive
gdrive.mount('/content/drive', force_remount=False)

DRIVE_ZIP_HINTS = [
    '/content/drive/MyDrive/ai_real_dataset_12k.zip',
    '/content/drive/MyDrive/ai-generated-images-vs-real-images.zip',
    '/content/drive/MyDrive/dataset.zip',
    '/content/drive/MyDrive/AI_vs_Real_Classifier/dataset.zip',
]

ZIP_PATH = next((h for h in DRIVE_ZIP_HINTS if os.path.exists(h)), None)

if ZIP_PATH is None:
    print('🔍 Scanning MyDrive for dataset zip...')
    for root_dir, _, files in os.walk('/content/drive/MyDrive'):
        for fname in files:
            if fname.endswith('.zip') and any(
                    kw in fname.lower() for kw in ('fake','real','ai','image','dataset','12k')):
                ZIP_PATH = os.path.join(root_dir, fname)
                print(f'  Found: {ZIP_PATH}')
                break
        if ZIP_PATH: break

if ZIP_PATH is None:
    raise FileNotFoundError(
        '\n❌ Dataset zip not found on Drive.\n'
        '   Add its path to DRIVE_ZIP_HINTS above.')

print(f'\n✅ Zip located : {ZIP_PATH}')
print(f'   Size        : {os.path.getsize(ZIP_PATH)/1024**3:.2f} GB')
print(f'   (Drive copy will NOT be touched)')


Mounted at /content/drive

✅ Zip located : /content/drive/MyDrive/ai_real_dataset_12k.zip
   Size        : 9.62 GB
   (Drive copy will NOT be touched)


In [ ]:
# ── Step 4: Scan zip, balance, split, extract ──────────────────────────────
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.webp')

def is_image(p): return p.lower().endswith(IMG_EXTS)

def detect_class(fpath):
    parts = fpath.replace('\\', '/').split('/')
    for p in parts[:-1]:
        lp = p.lower()
        if lp == 'real': return 'real'
        if lp in ('fake','ai','artificial','generated','synthetic'): return 'ai'
    base = parts[-1].lower()
    if 'real' in base: return 'real'
    if any(k in base for k in ('fake','ai_gen')): return 'ai'
    return None

print('📋 Scanning zip (no extraction yet)...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    all_entries = zf.namelist()

image_entries = [e for e in all_entries
                 if is_image(e) and not e.startswith('__MACOSX')]
print(f'   Image entries: {len(image_entries):,}')

categorized = {'ai': [], 'real': []}
for fp in image_entries:
    cls = detect_class(fp)
    if cls: categorized[cls].append(fp)

for cls, files in categorized.items():
    print(f'   {cls}: {len(files):,}')

# Balance
min_count = min(len(v) for v in categorized.values())
for cls in categorized:
    random.shuffle(categorized[cls])
    categorized[cls] = categorized[cls][:min_count]

n_train = int(min_count * TRAIN_FRAC)
n_val   = int(min_count * VAL_FRAC)
splits = {
    'train': {c: categorized[c][:n_train]              for c in categorized},
    'val':   {c: categorized[c][n_train:n_train+n_val] for c in categorized},
    'test':  {c: categorized[c][n_train+n_val:]        for c in categorized},
}
print('\n📊 Split plan:')
for sp, cls in splits.items():
    total = sum(len(v) for v in cls.values())
    print(f'   {sp:5s}: {total:,}')

# Directories
for sp in splits:
    for cls in ['ai','real']:
        os.makedirs(f'{DATASET_ROOT}/{sp}/{cls}', exist_ok=True)

# Skip if already extracted
already_done = all(
    os.path.exists(f'{DATASET_ROOT}/{sp}/{cls}') and
    len(os.listdir(f'{DATASET_ROOT}/{sp}/{cls}')) > 10
    for sp in splits for cls in ['ai','real']
)

if already_done:
    print('\n✅ Already extracted — skipping.')
else:
    from tqdm import tqdm
    print('\n🚀 Extracting (Drive zip is read-only)...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for sp, classes in splits.items():
            for cls, file_list in classes.items():
                dst_dir = f'{DATASET_ROOT}/{sp}/{cls}'
                for i, zpath in enumerate(
                        tqdm(file_list, desc=f'{sp}/{cls}', unit='img')):
                    dst = os.path.join(dst_dir,
                                       f'{i:06d}_{os.path.basename(zpath)}')
                    if os.path.exists(dst): continue
                    try:
                        with zf.open(zpath) as src: data = src.read()
                        Image.open(io.BytesIO(data)).verify()
                        with open(dst, 'wb') as f: f.write(data)
                    except Exception: pass
    print('\n✅ Extraction complete.')

# Count
total = 0
for sp in ['train','val','test']:
    for cls in ['ai','real']:
        n = len([f for f in os.listdir(f'{DATASET_ROOT}/{sp}/{cls}')
                 if is_image(f)])
        total += n
        print(f'   {sp}/{cls}: {n:,}')
print(f'   Grand total: {total:,}')


📋 Scanning zip (no extraction yet)...
   Image entries: 12,000
   ai: 6,000
   real: 6,000

📊 Split plan:
   train: 9,000
   val  : 1,800
   test : 1,200

🚀 Extracting (Drive zip is read-only)...


test/real: 100%|██████████| 600/600 [00:10<00:00, 58.34img/s]


✅ Extraction complete.
   train/ai: 4,500
   train/real: 4,500
   val/ai: 900
   val/real: 899
   test/ai: 600
   test/real: 600
   Grand total: 11,999


In [ ]:
# ── Step 5: tf.data pipeline (replaces ImageDataGenerator) ─────────────────
#
# WHY tf.data vs ImageDataGenerator:
#   - Parallel map + prefetch overlaps CPU preprocessing with GPU training
#   - XLA-compatible augmentation ops
#   - cache=False: avoids OOM on large datasets / Colab free-tier RAM

# ── Collect file paths + labels ──────────────────────────────────────────────
def collect_paths(split):
    paths, labels = [], []
    for label, cls in enumerate(['ai', 'real']):
        d = f'{DATASET_ROOT}/{split}/{cls}'
        for fname in os.listdir(d):
            if is_image(fname):
                paths.append(os.path.join(d, fname))
                labels.append(float(label))
    idx = list(range(len(paths)))
    random.shuffle(idx)
    return [paths[i] for i in idx], [labels[i] for i in idx]

train_paths, train_labels = collect_paths('train')
val_paths,   val_labels   = collect_paths('val')
test_paths,  test_labels  = collect_paths('test')
print(f'Train: {len(train_paths):,}  Val: {len(val_paths):,}  Test: {len(test_paths):,}')

# ── Image decode + resize (robust: handles JPEG / PNG / WebP + corrupt files)
def load_image(path, label):
    raw = tf.io.read_file(path)
    # decode_image handles JPEG/PNG/WebP/GIF automatically
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])   # needed after decode_image
    return img, label

# ── Phase 1 augmentation (lightweight) ──────────────────────────────────────
def augment_p1(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    return img, label

# ── Phase 2 augmentation (heavy real-world) ──────────────────────────────────
def augment_p2(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.3)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    img = tf.image.random_saturation(img, 0.7, 1.3)
    img = tf.image.random_hue(img, 0.05)

    shape  = tf.shape(img)
    crop_h = tf.cast(tf.cast(shape[0], tf.float32) * tf.random.uniform([], 0.85, 1.0), tf.int32)
    crop_w = tf.cast(tf.cast(shape[1], tf.float32) * tf.random.uniform([], 0.85, 1.0), tf.int32)
    img    = tf.image.random_crop(img, size=[crop_h, crop_w, 3])
    img    = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=0.02)
    img   = tf.clip_by_value(img + noise, 0.0, 1.0)

    img_uint8 = tf.cast(img * 255.0, tf.uint8)
    quality   = tf.random.uniform([], 50, 95, dtype=tf.int32)
    img_jpeg  = tf.image.adjust_jpeg_quality(img_uint8, quality)
    img       = tf.cast(img_jpeg, tf.float32) / 255.0
    img       = tf.clip_by_value(img, 0.0, 1.0)

    return img, label

# ── Build tf.data datasets (cache=False — avoids OOM, skips corrupt files) ──
def make_dataset(paths, labels, augment_fn=None, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED,
                        reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    # ↓ silently drop any corrupt images that cause decode errors
    ds = ds.apply(tf.data.experimental.ignore_errors())
    if augment_fn:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds_p1 = make_dataset(train_paths, train_labels, augment_fn=augment_p1)
train_ds_p2 = make_dataset(train_paths, train_labels, augment_fn=augment_p2)
val_ds      = make_dataset(val_paths,   val_labels,   augment_fn=None, shuffle=False)
test_ds     = make_dataset(test_paths,  test_labels,  augment_fn=None, shuffle=False)

print('✅ tf.data pipelines ready (no cache — RAM safe)')
print(f'   Train samples : {len(train_paths):,}')
print(f'   Val samples   : {len(val_paths):,}')

Train: 9,000  Val: 1,799  Test: 1,200


Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


✅ tf.data pipelines ready (no cache — RAM safe)
   Train samples : 9,000
   Val samples   : 1,799


In [ ]:
# ── Step 6: Build EfficientNetV2-S model ────────────────────────────────────
#
# WHY EfficientNetV2-S over EfficientNetB3:
#   - Fused-MBConv blocks train 2-3× faster on GPU
#   - Better accuracy at same parameter count (~20M params)
#   - Progressive learning aware — handles varied augmentation better
#   - Native 224px support
# ─────────────────────────────────────────────────────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2S

def build_model(trainable_base=False, unfreeze_top=0):
    base = EfficientNetV2S(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_preprocessing=True   # built-in normalisation — no manual rescale needed
    )
    base.trainable = trainable_base
    if trainable_base and unfreeze_top > 0:
        for layer in base.layers[:-unfreeze_top]:
            layer.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32)
    x = base(inputs, training=False)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.45)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)

    # Output in float32 — required for numerical stability with mixed precision
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)

    return Model(inputs, outputs, name='EffNetV2S_AIvsReal'), base

model, base = build_model(trainable_base=False)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_P1, clipnorm=1.0),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

trainable_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'✅ Model built: {model.name}')
print(f'   Trainable params : {trainable_params:,}')
print(f'   Total layers     : {len(model.layers)}')


82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ Model built: EffNetV2S_AIvsReal
   Trainable params : 791,553
   Total layers     : 11


In [ ]:
# # ── Step 7: Phase 1 — Train head only (base frozen) ────────────────────────
# import math

# CKPT_P1 = f'{CHECKPOINT_DIR}/ckpt_p1.keras'

# # Cosine decay LR schedule with warm-up
# steps_per_epoch = math.ceil(len(train_paths) / BATCH_SIZE)
# total_steps_p1  = EPOCHS_P1 * steps_per_epoch
# warmup_steps    = 1 * steps_per_epoch   # 1-epoch linear warm-up

# lr_schedule_p1 = keras.optimizers.schedules.CosineDecayRestarts(
#     initial_learning_rate=LR_P1,
#     first_decay_steps=total_steps_p1,
#     t_mul=1.0, m_mul=0.9, alpha=1e-6
# )

# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=lr_schedule_p1, clipnorm=1.0),
#     loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
#     metrics=['accuracy', keras.metrics.AUC(name='auc'),
#              keras.metrics.Precision(name='precision'),
#              keras.metrics.Recall(name='recall')]
# )

# callbacks_p1 = [
#     keras.callbacks.EarlyStopping(
#         monitor='val_auc', patience=4,
#         restore_best_weights=True, min_delta=0.001, verbose=1),
#     keras.callbacks.ModelCheckpoint(
#         CKPT_P1, monitor='val_auc',
#         save_best_only=True, verbose=1),
#     keras.callbacks.ReduceLROnPlateau(
#         monitor='val_loss', factor=0.4, patience=2,
#         min_lr=1e-7, verbose=1),
#     keras.callbacks.TensorBoard(
#         log_dir='/content/logs/phase1', histogram_freq=0),
# ]

# print(f'🚀 Phase 1: head-only training — max {EPOCHS_P1} epochs')
# print(f'   Steps/epoch: {steps_per_epoch}')

# history_p1 = model.fit(
#     train_ds_p1,
#     epochs=EPOCHS_P1,
#     validation_data=val_ds,
#     callbacks=callbacks_p1,
#     verbose=1
# )

# best_p1_acc = max(history_p1.history['val_accuracy'])
# best_p1_auc = max(history_p1.history['val_auc'])
# print(f'\n✅ Phase 1 done | val_acc={best_p1_acc:.4f} | val_AUC={best_p1_auc:.4f}')

In [ ]:
# ── Step 7: Phase 1 — Train head only (base frozen) RUN THIS NEXT TIME!!!!!
import math

CKPT_P1 = f'{CHECKPOINT_DIR}/ckpt_p1.keras'

# Cosine decay LR schedule with warm-up
steps_per_epoch = math.ceil(len(train_paths) / BATCH_SIZE)
total_steps_p1  = EPOCHS_P1 * steps_per_epoch
warmup_steps    = 1 * steps_per_epoch   # 1-epoch linear warm-up

lr_schedule_p1 = keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=LR_P1,
    first_decay_steps=total_steps_p1,
    t_mul=1.0, m_mul=0.9, alpha=1e-6
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule_p1, clipnorm=1.0),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=['accuracy', keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

callbacks_p1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=4,
        restore_best_weights=True, min_delta=0.001, verbose=1),
    keras.callbacks.ModelCheckpoint(
        CKPT_P1, monitor='val_auc',
        save_best_only=True, verbose=1),
    keras.callbacks.TensorBoard(
        log_dir='/content/logs/phase1', histogram_freq=0),
]

print(f'🚀 Phase 1: head-only training — max {EPOCHS_P1} epochs')
print(f'   Steps/epoch: {steps_per_epoch}')

history_p1 = model.fit(
    train_ds_p1,
    epochs=EPOCHS_P1,
    validation_data=val_ds,
    callbacks=callbacks_p1,
    verbose=1
)

best_p1_acc = max(history_p1.history['val_accuracy'])
best_p1_auc = max(history_p1.history['val_auc'])
print(f'\n✅ Phase 1 done | val_acc={best_p1_acc:.4f} | val_AUC={best_p1_auc:.4f}')

🚀 Phase 1: head-only training — max 10 epochs
   Steps/epoch: 71
Epoch 1/10
     71/Unknown 693s 8s/step - accuracy: 0.5845 - auc: 0.6131 - loss: 0.8224 - precision: 0.5844 - recall: 0.5752
Epoch 1: val_auc improved from None to 0.71229, saving model to /content/checkpoints/ckpt_p1.keras

Epoch 1: finished saving model to /content/checkpoints/ckpt_p1.keras
71/71 ━━━━━━━━━━━━━━━━━━━━ 867s 10s/step - accuracy: 0.6055 - auc: 0.6416 - loss: 0.7718 - precision: 0.6080 - recall: 0.5939 - val_accuracy: 0.5189 - val_auc: 0.7123 - val_loss: 0.7005 - val_precision: 0.9459 - val_recall: 0.0390
Epoch 2/10
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.6320 - auc: 0.6685 - loss: 0.7118 - precision: 0.6398 - recall: 0.6165
Epoch 2: val_auc improved from 0.71229 to 0.75094, saving model to /content/checkpoints/ckpt_p1.keras

Epoch 2: finished saving model to /content/checkpoints/ckpt_p1.keras
71/71 ━━━━━━━━━━━━━━━━━━━━ 615s 9s/step - accuracy: 0.6337 - auc: 0.6736 - loss: 0.7013 - precision: 0.6

In [ ]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create folder in Drive (if not exists)
DRIVE_DIR = '/content/drive/MyDrive/ai_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy checkpoint to Drive
shutil.copy(CKPT_P1, f'{DRIVE_DIR}/ckpt_p1.keras')

print('✅ Checkpoint saved to Drive!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Checkpoint saved to Drive!


In [ ]:
from google.colab import drive
import shutil
import os

# Mount Drive
drive.mount('/content/drive')

# Path to your saved checkpoint in Drive
DRIVE_CKPT = '/content/drive/MyDrive/ai_checkpoints/ckpt_p1.keras'

# Local path (where your code expects it)
LOCAL_CKPT = f'{CHECKPOINT_DIR}/ckpt_p1.keras'

# Ensure checkpoint folder exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Copy from Drive → Colab
shutil.copy(DRIVE_CKPT, LOCAL_CKPT)

print('✅ Checkpoint loaded from Drive')

# Load weights into model
model.load_weights(LOCAL_CKPT)

print('✅ Weights restored into model')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Checkpoint loaded from Drive
✅ Weights restored into model


In [ ]:
# # ── Step 8: Phase 2 — Fine-tune top layers ──────────────────────────────────
# CKPT_BEST = f'{CHECKPOINT_DIR}/ckpt_best.keras'

# # Load best Phase 1 weights
# if os.path.exists(CKPT_P1):
#     model.load_weights(CKPT_P1)
#     print(f'✅ Loaded Phase 1 best weights')

# # Unfreeze top-N base layers
# base.trainable = True
# for layer in base.layers[:-UNFREEZE_TOP]:
#     layer.trainable = False

# n_trainable = sum(1 for l in base.layers if l.trainable)
# n_frozen    = sum(1 for l in base.layers if not l.trainable)
# print(f'   Base — trainable: {n_trainable} | frozen: {n_frozen}')

# # Cosine decay for Phase 2
# total_steps_p2 = EPOCHS_P2 * steps_per_epoch
# lr_schedule_p2 = keras.optimizers.schedules.CosineDecay(
#     initial_learning_rate=LR_P2,
#     decay_steps=total_steps_p2,
#     alpha=1e-7
# )

# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=lr_schedule_p2, clipnorm=1.0),
#     loss=keras.losses.BinaryCrossentropy(label_smoothing=0.03),
#     metrics=['accuracy', keras.metrics.AUC(name='auc'),
#              keras.metrics.Precision(name='precision'),
#              keras.metrics.Recall(name='recall')]
# )

# callbacks_p2 = [
#     keras.callbacks.EarlyStopping(
#         monitor='val_auc', patience=4,
#         restore_best_weights=True, verbose=1),
#     keras.callbacks.ModelCheckpoint(
#         CKPT_BEST, monitor='val_auc',
#         save_best_only=True, verbose=1),
#     keras.callbacks.TensorBoard(
#         log_dir='/content/logs/phase2', histogram_freq=0),
#     # ── Crash-safe: save every epoch so runtime resets lose at most 1 epoch
#     keras.callbacks.ModelCheckpoint(
#         f'{CHECKPOINT_DIR}/epoch_{{epoch:02d}}.keras',
#         save_best_only=False, verbose=0),
# ]

# print(f'\n🚀 Phase 2: fine-tuning top {UNFREEZE_TOP} layers — max {EPOCHS_P2} epochs')
# history_p2 = model.fit(
#     train_ds_p2,
#     epochs=EPOCHS_P2,
#     validation_data=val_ds,
#     callbacks=callbacks_p2,
#     verbose=1
# )

# best_p2_acc = max(history_p2.history['val_accuracy'])
# best_p2_auc = max(history_p2.history['val_auc'])
# print(f'\n✅ Phase 2 done | val_acc={best_p2_acc:.4f} | val_AUC={best_p2_auc:.4f}')

In [ ]:
# ── Step 8: Phase 2 — Fine-tune top layers (v3 FINAL FIXED) ───────────────

import os, math, numpy as np, tensorflow as tf
from tensorflow import keras

CKPT_P1   = f'{CHECKPOINT_DIR}/ckpt_p1.keras'
CKPT_BEST = f'{CHECKPOINT_DIR}/ckpt_best.keras'

# ── Tuned hyperparameters ─────────────────────────────────────────────────
UNFREEZE_TOP = 40
LR_P2_PEAK   = 2e-5
LR_P2_WARMUP = 5e-6
EPOCHS_P2    = 12
LABEL_SMOOTH = 0.02

steps_per_epoch = math.ceil(len(train_paths) / BATCH_SIZE)

# ── Load Phase 1 weights ──────────────────────────────────────────────────
if os.path.exists(CKPT_P1):
    model.load_weights(CKPT_P1)
    print('✅ Loaded Phase 1 best weights')
else:
    raise FileNotFoundError('❌ CKPT_P1 not found')

# ── Unfreeze strategy ─────────────────────────────────────────────────────
base.trainable = True
for layer in base.layers[:-UNFREEZE_TOP]:
    layer.trainable = False

# 🔴 Freeze BatchNorm layers (VERY IMPORTANT)
for layer in base.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

n_trainable = sum(1 for l in base.layers if l.trainable)
n_frozen    = sum(1 for l in base.layers if not l.trainable)
print(f'✅ Base — trainable: {n_trainable} | frozen: {n_frozen}')

# ── Warmup + Cosine LR ────────────────────────────────────────────────────
warmup_epochs  = 2
warmup_steps   = warmup_epochs * steps_per_epoch
total_steps_p2 = EPOCHS_P2 * steps_per_epoch
decay_steps    = total_steps_p2 - warmup_steps

class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, warmup_steps, peak_lr, decay_steps, alpha=1e-7):
        super().__init__()
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.peak_lr      = tf.cast(peak_lr, tf.float32)
        self.decay_steps  = tf.cast(decay_steps, tf.float32)
        self.alpha        = tf.cast(alpha, tf.float32)
        self._warmup_lr   = tf.cast(LR_P2_WARMUP, tf.float32)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)

        # Warmup
        warmup_lr = self._warmup_lr + (self.peak_lr - self._warmup_lr) * (step / self.warmup_steps)

        # Cosine decay
        cos_step  = tf.maximum(step - self.warmup_steps, 0.0)
        cos_decay = 0.5 * (1.0 + tf.cos(
            np.pi * tf.minimum(cos_step, self.decay_steps) / self.decay_steps
        ))
        decay_lr = self.alpha + (self.peak_lr - self.alpha) * cos_decay

        return tf.where(step < self.warmup_steps, warmup_lr, decay_lr)

    def get_config(self):
        return {
            'warmup_steps': int(self.warmup_steps.numpy()),
            'peak_lr': float(self.peak_lr.numpy()),
            'decay_steps': int(self.decay_steps.numpy()),
            'alpha': float(self.alpha.numpy()),
        }

lr_schedule_p2 = WarmupCosineDecay(
    warmup_steps=warmup_steps,
    peak_lr=LR_P2_PEAK,
    decay_steps=decay_steps,
)

# ── Compile ───────────────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=lr_schedule_p2,
        clipnorm=1.0,
        epsilon=1e-7
    ),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)

# ── Blended dataset (smart augmentation) ──────────────────────────────────
train_ds_blend = tf.data.Dataset.sample_from_datasets(
    [train_ds_p1, train_ds_p2],
    weights=[0.4, 0.6],
    seed=SEED
)

# ── Callbacks ─────────────────────────────────────────────────────────────
callbacks_p2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=5,
        min_delta=0.001,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        CKPT_BEST,
        monitor='val_auc',
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.TensorBoard(
        log_dir='/content/logs/phase2',
        histogram_freq=0
    ),

    # Crash-safe: save every epoch
    keras.callbacks.ModelCheckpoint(
        f'{CHECKPOINT_DIR}/epoch_{{epoch:02d}}.keras',
        save_best_only=False,
        verbose=0
    ),

    # ✅ FIXED LR logger — no longer callable error
    keras.callbacks.LambdaCallback(
        on_epoch_end=lambda epoch, logs: print(
            f'  📉 LR: {float(keras.backend.get_value(model.optimizer.learning_rate)):.2e}'
        )
    ),
]

# ── Train ─────────────────────────────────────────────────────────────────
print(f'\n🚀 Phase 2 (v3): fine-tuning top {UNFREEZE_TOP} layers')
print(f'📊 Steps/epoch: {steps_per_epoch}')
print(f'🌡️ Warmup: {warmup_epochs} epochs → peak LR {LR_P2_PEAK:.0e}')

history_p2 = model.fit(
    train_ds_blend,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS_P2,
    validation_data=val_ds,
    callbacks=callbacks_p2,
    verbose=1
)

# ── Results ───────────────────────────────────────────────────────────────
best_p2_acc = max(history_p2.history['val_accuracy'])
best_p2_auc = max(history_p2.history['val_auc'])

print(f'\n✅ Phase 2 done | val_acc={best_p2_acc:.4f} | val_AUC={best_p2_auc:.4f}')
print(f'   Δ AUC vs Phase 1: {(best_p2_auc - 0.7815)*100:+.2f}pp')

# ── Backup to Drive ───────────────────────────────────────────────────────
import shutil
DRIVE_BACKUP = '/content/drive/MyDrive/AI_vs_Real_Classifier'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
shutil.copy(CKPT_BEST, f'{DRIVE_BACKUP}/ckpt_best_v3.keras')
print('💾 Best checkpoint backed up to Drive')

✅ Loaded Phase 1 best weights
✅ Base — trainable: 32 | frozen: 481

🚀 Phase 2 (v3): fine-tuning top 40 layers
📊 Steps/epoch: 71
🌡️ Warmup: 2 epochs → peak LR 2e-05
Epoch 1/12
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.6756 - auc: 0.7337 - loss: 0.6157 - precision: 0.6775 - recall: 0.6633
Epoch 1: val_auc improved from None to 0.77951, saving model to /content/checkpoints/ckpt_best.keras

Epoch 1: finished saving model to /content/checkpoints/ckpt_best.keras
  📉 LR: 1.25e-05
71/71 ━━━━━━━━━━━━━━━━━━━━ 837s 10s/step - accuracy: 0.6769 - auc: 0.7387 - loss: 0.6117 - precision: 0.6824 - recall: 0.6629 - val_accuracy: 0.7119 - val_auc: 0.7795 - val_loss: 0.5782 - val_precision: 0.7778 - val_recall: 0.5924
Epoch 2/12
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.6793 - auc: 0.7441 - loss: 0.6054 - precision: 0.6834 - recall: 0.6617
Epoch 2: val_auc improved from 0.77951 to 0.78310, saving model to /content/checkpoints/ckpt_best.keras

Epoch 2: finished saving model to /content

In [ ]:
# ── Step 8b: RESUME after runtime crash (run this cell if session resets) ──
# ─────────────────────────────────────────────────────────────────────────────
# 1. Re-run cells 1-6 to restore imports, config, data pipeline, and model
# 2. Then run THIS cell to resume Phase 2 from the last checkpoint
# ─────────────────────────────────────────────────────────────────────────────

RESUME = False   # ← set True only after a crash; then run this cell

if RESUME:
    # Find latest epoch checkpoint
    ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/epoch_*.keras'))
    if ckpts:
        latest = ckpts[-1]
        print(f'Resuming from: {latest}')
        model, base = build_model(trainable_base=True, unfreeze_top=UNFREEZE_TOP)
        model.load_weights(latest)
        start_epoch = int(os.path.basename(latest).split('_')[1].split('.')[0])
    elif os.path.exists(CKPT_BEST):
        print(f'Resuming from best checkpoint: {CKPT_BEST}')
        model, base = build_model(trainable_base=True, unfreeze_top=UNFREEZE_TOP)
        model.load_weights(CKPT_BEST)
        start_epoch = 0
    else:
        print('No checkpoint found — start from Phase 1')
        start_epoch = 0

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR_P2, clipnorm=1.0),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=0.03),
        metrics=['accuracy', keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )

    remaining = max(1, EPOCHS_P2 - start_epoch)
    print(f'Resuming {remaining} remaining epochs from epoch {start_epoch}')

    history_resume = model.fit(
        train_ds_p2,
        initial_epoch=start_epoch,
        epochs=start_epoch + remaining,
        validation_data=val_ds,
        callbacks=callbacks_p2,
        verbose=1
    )
    print('✅ Resumed training complete')
else:
    print('RESUME=False — skipping. Set RESUME=True only after a crash.')


In [ ]:
# ── Step 9: Evaluate on validation + test sets ──────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, roc_curve, auc)

# Load best weights
if os.path.exists(CKPT_BEST):
    model.load_weights(CKPT_BEST)
    print(f'✅ Loaded best weights: {CKPT_BEST}')

def evaluate_split(ds, name='val'):
    y_prob = model.predict(ds, verbose=1).flatten()
    y_pred = (y_prob > 0.5).astype(int)
    y_true = np.concatenate([y.numpy() for _, y in ds]).astype(int)
    acc  = accuracy_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    print(f'\n🏆 {name.upper()} Results')
    print(f'   Accuracy : {acc:.4f} ({acc*100:.2f}%)')
    print(f'   AUC      : {roc_auc:.4f}')
    print(classification_report(y_true, y_pred, target_names=['AI','REAL']))
    return dict(y_true=y_true, y_pred=y_pred, y_prob=y_prob,
                acc=acc, auc=roc_auc, fpr=fpr, tpr=tpr)

val_m  = evaluate_split(val_ds,  'Validation')
test_m = evaluate_split(test_ds, 'Test')

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('EfficientNetV2-S  —  Evaluation', fontsize=13, fontweight='bold')

# Confusion matrix
cm = confusion_matrix(val_m['y_true'], val_m['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['AI','REAL'], yticklabels=['AI','REAL'],
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix (Val)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

# ROC curve
axes[1].plot(val_m['fpr'],  val_m['tpr'],  'b-', lw=2,
             label=f'Val  AUC={val_m["auc"]:.4f}')
axes[1].plot(test_m['fpr'], test_m['tpr'], 'r--', lw=2,
             label=f'Test AUC={test_m["auc"]:.4f}')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')

# Training curves (Phase 2)
ep = range(1, len(history_p2.history['accuracy'])+1)
axes[2].plot(ep, history_p2.history['val_accuracy'], 'b-',  label='Val Acc')
axes[2].plot(ep, history_p2.history['accuracy'],     'b--', label='Train Acc', alpha=0.6)
axes[2].plot(ep, history_p2.history['val_auc'],      'r-',  label='Val AUC')
axes[2].set_title('Phase 2 Curves'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/evaluation_v2.png', dpi=130, bbox_inches='tight')
plt.show()
print('\n📊 Plots saved → /content/evaluation_v2.png')

gap = abs(val_m['acc'] - test_m['acc'])
print(f'\nVal↔Test gap: {gap*100:.2f}%  '
      f'{\'✅ Good generalisation\' if gap < 0.05 else \'⚠️  >5% — check for overfitting\'}')


In [ ]:
# ── Step 10: TTA inference function ─────────────────────────────────────────
def preprocess_for_inference(image_source, apply_aug=False):
    if isinstance(image_source, str):
        img = Image.open(image_source).convert('RGB')
    else:
        img = image_source.copy().convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    if apply_aug:
        if random.random() > 0.5: arr = arr[:, ::-1, :]          # h-flip
        if random.random() > 0.7: arr = np.clip(arr * random.uniform(0.9, 1.1), 0, 255)
    return arr / 255.0

def predict_image(image_source, tta_steps=8, threshold=THRESHOLD, verbose=True):
    """
    Classify one image as AI or Real using TTA.

    Parameters
    ----------
    image_source : str | PIL.Image
    tta_steps    : int    — 1 for fast single-pass, 8 for best accuracy
    threshold    : float  — below this → UNCERTAIN
    verbose      : bool

    Returns
    -------
    dict: {label, confidence, ai_prob, real_prob, tta_steps}
    """
    probs = []
    for i in range(tta_steps):
        arr  = preprocess_for_inference(image_source, apply_aug=(i > 0))
        arr  = np.expand_dims(arr, 0)
        prob = float(model.predict(arr, verbose=0)[0][0])
        probs.append(prob)

    real_p = float(np.mean(probs))
    ai_p   = 1.0 - real_p

    if   real_p >= threshold: label, conf = 'REAL',      real_p
    elif ai_p   >= threshold: label, conf = 'AI',        ai_p
    else:                     label, conf = 'UNCERTAIN',  max(real_p, ai_p)

    result = dict(label=label, confidence=round(conf,4),
                  ai_prob=round(ai_p,4), real_prob=round(real_p,4),
                  tta_steps=tta_steps)
    if verbose:
        emoji = {'AI':'🤖','REAL':'📷','UNCERTAIN':'❓'}[label]
        print(f'\n{emoji}  {label}  (confidence: {conf:.1%})')
        print(f'   AI prob  : {ai_p:.1%}')
        print(f'   Real prob: {real_p:.1%}')
        print(f'   TTA steps: {tta_steps}')
        if label == 'UNCERTAIN':
            print('   ⚠️  Below threshold — treat with caution')
    return result

print(f'✅ predict_image() ready | threshold={THRESHOLD:.0%} | TTA={8}')


In [ ]:
# ── Step 11: Quick visual test — 6 random validation images ────────────────
from PIL import Image
import random as pyrandom

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
sample_paths  = pyrandom.sample(val_paths, 6)
sample_labels = [int(val_labels[val_paths.index(p)]) for p in sample_paths]
label_map     = {0: 'AI', 1: 'REAL'}

for ax, path, true_label_idx in zip(axes.flat, sample_paths, sample_labels):
    true_label = label_map[true_label_idx]
    result     = predict_image(path, tta_steps=8, verbose=False)
    img        = Image.open(path).resize((224, 224))
    ax.imshow(img)
    color = 'green' if result['label'] == true_label else 'red'
    ax.set_title(
        f'True: {true_label} | Pred: {result["label"]}\n'
        f'Conf: {result["confidence"]:.0%}',
        color=color, fontsize=10
    )
    ax.axis('off')

plt.suptitle('Visual Test — Green = Correct, Red = Wrong', fontsize=12)
plt.tight_layout()
plt.savefig('/content/visual_test_v2.png', dpi=100)
plt.show()
print('✅ Visual test complete')


In [ ]:
# ── Step 12: Test your own image ─────────────────────────────────────────────
print('📤 Upload any image (Cancel to skip):')
try:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    for fname, data in uploaded.items():
        with open(fname, 'wb') as f: f.write(data)
        print(f'\n🔍 Testing: {fname}')
        result = predict_image(fname, tta_steps=8)
        img = Image.open(fname)
        plt.figure(figsize=(5, 5))
        plt.imshow(np.array(img))
        color = {'AI':'red','REAL':'green','UNCERTAIN':'orange'}[result['label']]
        plt.title(
            f"{result['label']} ({result['confidence']:.1%})\n"
            f"AI={result['ai_prob']:.0%}  REAL={result['real_prob']:.0%}",
            color=color, fontsize=13, fontweight='bold'
        )
        plt.axis('off'); plt.tight_layout(); plt.show()
except Exception as e:
    print(f'Skipped: {e}')


In [ ]:
# ── Step 13: Save model_v2.keras + class_names_v2.json ─────────────────────
from tensorflow import keras

# Load best weights before saving
if os.path.exists(CKPT_BEST):
    model.load_weights(CKPT_BEST)

# Save as .keras (modern format — faster load, smaller than .h5)
model.save(MODEL_OUT)
size_mb = os.path.getsize(MODEL_OUT) / (1024**2)
print(f'✅ model_v2.keras saved ({size_mb:.1f} MB) → {MODEL_OUT}')

# Save class config
config = {
    'class_names'         : ['ai', 'real'],
    'class_indices'       : {'ai': 0, 'real': 1},
    'label_map'           : {'0': 'AI', '1': 'REAL'},
    'confidence_threshold': THRESHOLD,
    'model_architecture'  : 'EfficientNetV2S',
    'input_size'          : IMG_SIZE,
    'tta_steps_default'   : 8,
    'include_preprocessing': True,
    'final_metrics': {
        'val_accuracy' : round(float(val_m['acc']),  4),
        'val_auc'      : round(float(val_m['auc']),  4),
        'test_accuracy': round(float(test_m['acc']), 4),
        'test_auc'     : round(float(test_m['auc']), 4),
    },
    'dataset': {
        'source'        : 'Google Drive zip (read-only — never deleted)',
        'total_images'  : len(train_paths) + len(val_paths) + len(test_paths),
        'train'         : len(train_paths),
        'val'           : len(val_paths),
        'test'          : len(test_paths),
        'train_frac'    : TRAIN_FRAC,
        'val_frac'      : VAL_FRAC,
    },
    'training': {
        'framework'         : 'TensorFlow/Keras',
        'mixed_precision'   : 'float16',
        'xla_jit'           : True,
        'data_pipeline'     : 'tf.data',
        'batch_size'        : BATCH_SIZE,
        'phase1_lr'         : LR_P1,
        'phase2_lr'         : LR_P2,
        'phase2_unfreeze_top': UNFREEZE_TOP,
    }
}

with open(CLASS_JSON, 'w') as f:
    json.dump(config, f, indent=2)
print(f'✅ class_names_v2.json saved')

# Backup to Drive
DRIVE_BACKUP = '/content/drive/MyDrive/AI_vs_Real_Classifier'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
for src in [MODEL_OUT, CLASS_JSON,
            '/content/evaluation_v2.png',
            '/content/visual_test_v2.png']:
    if os.path.exists(src):
        dst = f'{DRIVE_BACKUP}/{os.path.basename(src)}'
        shutil.copy2(src, dst)
        print(f'   💾 Backed up: {os.path.basename(src)} ({os.path.getsize(dst)/1024**2:.1f} MB)')

print('\n✅ All artifacts backed up to Drive (zip untouched)')

# Download
from google.colab import files as colab_files
for path in [MODEL_OUT, CLASS_JSON]:
    colab_files.download(path)
    print(f'✅ Downloaded: {os.path.basename(path)}')


In [ ]:
# ── Step 14: Standalone deployment snippet ──────────────────────────────────
# Copy this block to src/predict.py in your project
SNIPPET = '''
import json, random
import numpy as np
from PIL import Image
import tensorflow as tf

_model     = tf.keras.models.load_model('models/model_v2.keras')
_cfg       = json.load(open('models/class_names_v2.json'))
_threshold = _cfg['confidence_threshold']
_img_size  = _cfg['input_size']
_tta       = _cfg.get('tta_steps_default', 8)

# Note: include_preprocessing=True means NO manual /255 rescale needed
def predict(image_path: str, tta: bool = True) -> dict:
    img   = Image.open(image_path).convert('RGB').resize((_img_size, _img_size), Image.LANCZOS)
    steps = _tta if tta else 1
    probs = []
    for i in range(steps):
        arr = np.array(img, dtype=np.float32)
        if i > 0 and random.random() > 0.5:
            arr = arr[:, ::-1, :]           # random h-flip for TTA
        arr = np.expand_dims(arr, axis=0)   # do NOT divide by 255 — model does it
        probs.append(float(_model.predict(arr, verbose=0)[0][0]))
    real_p = float(np.mean(probs))
    ai_p   = 1.0 - real_p
    if real_p >= _threshold:  label, conf = 'REAL',     real_p
    elif ai_p >= _threshold:  label, conf = 'AI',       ai_p
    else:                     label, conf = 'UNCERTAIN', max(real_p, ai_p)
    return dict(label=label, confidence=round(conf,4),
                ai_prob=round(ai_p,4), real_prob=round(real_p,4))
'''
print(SNIPPET)
print('✅ Copy the block above into src/predict.py')
